In [4]:
import pandas as pd
import numpy as np
import yfinance as yf
import requests 
import os
from dotenv import load_dotenv
import logging
import time
from tqdm import tqdm
import pyarrow as pa
import pyarrow.parquet as pq

# Adjust path if running from notebooks directory
if os.getcwd().endswith('notebooks'):
    os.chdir('..')

# Load environment variables
load_dotenv()

from src.data_loader import FinancialDataLoader

In [5]:
# Globals
raw_data_dir = "data/raw"
proc_data_dir = "data/processed"

# SEC User-Agent now pulled from .env for security
sec_user_agent = os.getenv("SEC_USER_AGENT")
headers = {"User-Agent": sec_user_agent}

In [6]:
# Setting up logger
logger = logging.getLogger(__name__)
logging.basicConfig(level=logging.INFO, format = "%(asctime)s - %(levelname)s - %(message)s")

loader = FinancialDataLoader()

## 1. Bronze Layer: Raw Ingestion

In [7]:
# File downloader wrapper
def download_file(url: str, filename: str, headers: dict = {}) -> bool:
    return loader.download_file(url, filename, headers)

In [8]:
download_file("https://www.sec.gov/files/company_tickers.json", "company_tickers.json", headers)

2026-06-13 16:30:21,104 - INFO - company_tickers.json already exists. Skipping download.


True

In [9]:
df_tickers = pd.read_json("data/raw/company_tickers.json", orient = "index")
df_tickers["cik_str"] = df_tickers["cik_str"].astype(str).str.zfill(10)

display(df_tickers.head())

,cik_str,ticker,title
0,0001045810,NVDA,NVIDIA CORP
1,0001652044,GOOGL,Alphabet Inc.
2,0000320193,AAPL,Apple Inc.
3,0000789019,MSFT,MICROSOFT CORP
4,0001018724,AMZN,AMAZON COM INC


In [10]:
# Metrics fetcher wrapper
def fetch_single_comp_metrics(ticker: str) -> dict:
    return loader._fetch_single_comp_metrics(ticker)

In [11]:
# Testing function with a single ticker
fetch_single_comp_metrics("NVDA")

{'ticker': 'NVDA',
 'enterprise_value': 4925240115200,
 'forwardPE': 16.122227,
 'ev_to_ebitda': 29.757,
 'ebitda': 165514002432,
 'total_cash': 53171998720,
 'total_debt': 12814000128,
 'employee_count': 42000,
 'estimated_revenue': 253491003392,
 'sector': 'Technology',
 'industry': 'Semiconductors',
 'business_summary': "NVIDIA Corporation operates as a data center scale AI infrastructure company. The company operates through two segments, Compute & Networking, and Graphics segments. The Compute & Networking segment provides data center accelerated computing and networking platforms and artificial intelligence solutions and software, and automotive platforms and autonomous and electric vehicle solutions, including software. The Graphics segment offers GeForce GPUs for gaming and PCs; Quadro/NVIDIA RTX GPUs for enterprise workstation graphics. The company's products are used in gaming, professional visualization, data center, and automotive markets. The company sells its products to 

In [ ]:
# Master Table Builder wrapper
def build_csv_comps_table(raw_data_path: str, output_csv: str, chunk_size: int = 50, limit: int = None):
    return loader.build_raw_master_table(raw_data_path, output_csv, chunk_size, limit)

In [ ]:
# Process first 10 for validation
build_csv_comps_table("data/raw/company_tickers.json", "data/raw/master_metrics.csv", 50, limit=10)

## 2. Gold Layer: ML-Ready Universal Table

In [ ]:
# Creating the Universal ML-Ready Table (Leak-Free Architecture)
master_file = "data/raw/master_metrics.csv"
output_parquet = "data/processed/UNIVERSAL_training.parquet"

loader.build_universal_training_table(master_file, output_parquet)

In [12]:
print("Universal Training Table Sample:")
df_universal = pd.read_parquet("data/processed/UNIVERSAL_training.parquet")
display(df_universal.head())

Universal Training Table Sample:


,ticker,enterprise_value,forwardPE,ev_to_ebitda,ebitda,total_cash,total_debt,employee_count,estimated_revenue,sector,industry,business_summary
0,NVDA,5.170628e+12,17.027882,31.240,165514002432,5.317200e+10,1.281400e+10,42000.0,2.534910e+11,Technology,Semiconductors,NVIDIA Corporation operates as a data center s...
1,GOOGL,4.609101e+12,26.421965,28.572,161315995648,1.268400e+11,9.587600e+10,194668.0,4.224980e+11,Communication Services,Internet Content & Information,Alphabet Inc. offers various products and plat...
2,AAPL,4.551953e+12,32.155922,28.454,159975997440,6.850700e+10,8.471100e+10,166000.0,4.514420e+11,Technology,Consumer Electronics,"Apple Inc. designs, manufactures, and markets ..."
3,MSFT,3.156524e+12,21.645250,17.113,184457003008,7.822800e+10,1.254320e+11,228000.0,3.182730e+11,Technology,Software - Infrastructure,Microsoft Corporation develops and supports so...
4,AMZN,2.957284e+12,27.011812,18.974,155860992000,1.430890e+11,2.355400e+11,1575000.0,7.427760e+11,Consumer Cyclical,Internet Retail,"Amazon.com, Inc. engages in the retail sale of..."


In [13]:
# Check Dead Letter Queue for failed tickers
dlq_path = "data/processed/missed_tickers.csv"
if os.path.exists(dlq_path):
    df_missed = pd.read_csv(dlq_path)
    print(f"Missed Tickers: {df_missed['failed_tickers'].tolist()}")